# Foundry Evals Upskilling

Based on [this doc](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluate-agent).

In [ ]:
%pip install "azure-ai-projects>=2.4.0" azure-identity python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

True

In [ ]:
# Imports
import os
import time
from pprint import pprint

from dotenv import load_dotenv

from openai.types.evals.create_eval_jsonl_run_data_source_param import (
    CreateEvalJSONLRunDataSourceParam,
    SourceFileContent,
    SourceFileContentContent,
)
from openai.types.eval_create_params import DataSourceConfigCustom
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator


In [4]:
# Intent resolution sample
endpoint = os.environ[
        "AZURE_AI_PROJECT_ENDPOINT"
    ]  # Sample : https://<account_name>.services.ai.azure.com/api/projects/<project_name>
model_deployment_name = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")  # Sample : gpt-4o-mini

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as client,
):

    data_source_config = DataSourceConfigCustom(
        type="custom",
        item_schema={
            "type": "object",
            "properties": {
                "query": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
                "response": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
                "tool_definitions": {"anyOf": [{"type": "object"}, {"type": "array", "items": {"type": "object"}}]},
            },
            "required": ["query", "response"],
        },
        include_sample_schema=True,
    )

    testing_criteria = [
        TestingCriterionAzureAIEvaluator(
            type="azure_ai_evaluator",
            name="intent_resolution",
            evaluator_name="builtin.intent_resolution",
            initialization_parameters={"model": f"{model_deployment_name}"},
            data_mapping={
                "query": "{{item.query}}",
                "response": "{{item.response}}",
                "tool_definitions": "{{item.tool_definitions}}",
            },
        )
    ]

    print("Creating Evaluation")
    eval_object = client.evals.create(
        name="Test Intent Resolution Evaluator with inline data",
        data_source_config=data_source_config,
        testing_criteria=testing_criteria,  # type: ignore
    )
    print("Evaluation created")

    print("Get Evaluation by Id")
    eval_object_response = client.evals.retrieve(eval_object.id)
    print("Eval Run Response:")
    pprint(eval_object_response)

    # Success example - Intent is identified and understood and the response correctly resolves user intent
    success_query = "What are the opening hours of the Eiffel Tower?"
    success_response = "Opening hours of the Eiffel Tower are 9:00 AM to 11:00 PM."

    # Failure example - Even though intent is correctly identified, the response does not resolve the user intent
    failure_query = "What is the opening hours of the Eiffel Tower?"
    failure_response = (
        "Please check the official website for the up-to-date information on Eiffel Tower opening hours."
    )

    # Complex conversation example with tool calls
    complex_query = [
        {"role": "system", "content": "You are a friendly and helpful customer service agent."},
        {
            "createdAt": "2025-03-14T06:14:20Z",
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Hi, I need help with my order #123 status?",
                }
            ],
        },
    ]

    complex_response = [
        {
            "createdAt": "2025-03-14T06:14:30Z",
            "run_id": "0",
            "role": "assistant",
            "content": [
                {
                    "type": "tool_call",
                    "tool_call_id": "tool_call_001",
                    "name": "get_order",
                    "arguments": {"order_id": "123"},
                }
            ],
        },
        {
            "createdAt": "2025-03-14T06:14:35Z",
            "run_id": "0",
            "tool_call_id": "tool_call_001",
            "role": "tool",
            "content": [
                {
                    "type": "tool_result",
                    "tool_result": '{ "order": { "id": "123", "status": "shipped", "delivery_date": "2025-03-15" } }',
                }
            ],
        },
        {
            "createdAt": "2025-03-14T06:14:40Z",
            "run_id": "0",
            "role": "assistant",
            "content": [
                {
                    "type": "tool_call",
                    "tool_call_id": "tool_call_002",
                    "name": "get_tracking",
                    "arguments": {"order_id": "123"},
                }
            ],
        },
        {
            "createdAt": "2025-03-14T06:14:45Z",
            "run_id": "0",
            "tool_call_id": "tool_call_002",
            "role": "tool",
            "content": [
                {
                    "type": "tool_result",
                    "tool_result": '{ "tracking_number": "ABC123", "carrier": "UPS" }',
                }
            ],
        },
        {
            "createdAt": "2025-03-14T06:14:50Z",
            "run_id": "0",
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Your order #123 has been shipped and is expected to be delivered on March 15, 2025. The tracking number is ABC123 with UPS.",
                }
            ],
        },
    ]

    # Tool definitions for the complex example
    tool_definitions = [
        {
            "name": "get_order",
            "description": "Get the details of a specific order.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string", "description": "The order ID to get the details for."}
                },
            },
        },
        {
            "name": "get_tracking",
            "description": "Get tracking information for an order.",
            "parameters": {
                "type": "object",
                "properties": {"order_id": {"type": "string", "description": "The order ID to get tracking for."}},
            },
        },
    ]

    print("Creating Eval Run with Inline Data")
    eval_run_object = client.evals.runs.create(
        eval_id=eval_object.id,
        name="inline_data_run",
        metadata={"team": "eval-exp", "scenario": "inline-data-v1"},
        data_source=CreateEvalJSONLRunDataSourceParam(
            type="jsonl",
            source=SourceFileContent(
                type="file_content",
                content=[
                    # Example 1: Success case - simple string query and response
                    SourceFileContentContent(item={"query": success_query, "response": success_response}),
                    # Example 2: Failure case - simple string query and response
                    SourceFileContentContent(item={"query": failure_query, "response": failure_response}),
                    # Example 3: Complex conversation with tool calls and tool definitions
                    SourceFileContentContent(
                        item={
                            "query": complex_query,
                            "response": complex_response,
                            "tool_definitions": tool_definitions,
                        }
                    ),
                    # Example 4: Complex conversation without tool definitions
                    SourceFileContentContent(item={"query": complex_query, "response": complex_response}),
                ],
            ),
        ),
    )

    print("Eval Run created")
    pprint(eval_run_object)

    print("Get Eval Run by Id")
    eval_run_response = client.evals.runs.retrieve(run_id=eval_run_object.id, eval_id=eval_object.id)
    print("Eval Run Response:")
    pprint(eval_run_response)

    print("\n\n----Eval Run Output Items----\n\n")

    while True:
        run = client.evals.runs.retrieve(run_id=eval_run_response.id, eval_id=eval_object.id)
        if run.status in ("completed", "failed"):
            output_items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_object.id))
            pprint(output_items)
            print(f"Eval Run Status: {run.status}")
            print(f"Eval Run Report URL: {run.report_url}")
            break
        time.sleep(5)
        print("Waiting for eval run to complete...")

Creating Evaluation
Evaluation created
Get Evaluation by Id
Eval Run Response:
EvalRetrieveResponse(id='eval_162abe43c5524778acb4403313c3bf74', created_at=1789480531, data_source_config=EvalCustomDataSourceConfig(schema_={'item': {'type': 'object', 'properties': {'query': {'anyOf': [{'type': 'string'}, {'type': 'array', 'items': {'type': 'object'}}]}, 'response': {'anyOf': [{'type': 'string'}, {'type': 'array', 'items': {'type': 'object'}}]}, 'tool_definitions': {'anyOf': [{'type': 'object'}, {'type': 'array', 'items': {'type': 'object'}}]}}, 'required': ['query', 'response']}, 'sample': {'type': 'object', 'properties': {'output_text': {'type': 'string'}}}}, type='custom', item_schema={}, include_sample_schema=True), metadata={}, name='Test Intent Resolution Evaluator with inline data', object='eval', testing_criteria=[LabelModelGrader(input=None, labels=None, model=None, name='intent_resolution', passing_labels=None, type='azure_ai_evaluator', id='intent_resolution_801d3148-0ebd-453d-

In [6]:
# Task adherence
with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as client,
):
    print("Creating an OpenAI client from the AI Project client")

    data_source_config = DataSourceConfigCustom(
        type="custom",
        item_schema={
            "type": "object",
            "properties": {
                "query": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
                "response": {"anyOf": [{"type": "string"}, {"type": "array", "items": {"type": "object"}}]},
                "tool_definitions": {"anyOf": [{"type": "object"}, {"type": "array", "items": {"type": "object"}}]},
            },
            "required": ["query", "response"],
        },
        include_sample_schema=True,
    )

    testing_criteria = [
        TestingCriterionAzureAIEvaluator(
            type="azure_ai_evaluator",
            name="task_adherence",
            evaluator_name="builtin.task_adherence",
            initialization_parameters={"deployment_name": f"{model_deployment_name}"},
            data_mapping={
                "query": "{{item.query}}",
                "response": "{{item.response}}",
                "tool_definitions": "{{item.tool_definitions}}",
            },
        )
    ]

    print("Creating Evaluation")
    eval_object = client.evals.create(
        name="Test Task Adherence Evaluator with inline data",
        data_source_config=data_source_config,
        testing_criteria=testing_criteria,  # type: ignore
    )
    print("Evaluation created")

    print("Get Evaluation by Id")
    eval_object_response = client.evals.retrieve(eval_object.id)
    print("Eval Run Response:")
    pprint(eval_object_response)

    # Failure example - vague adherence to the task
    failure_query = "What are the best practices for maintaining a healthy rose garden during the summer?"
    failure_response = "Make sure to water your roses regularly and trim them occasionally."

    # Success example - full adherence to the task
    success_query = "What are the best practices for maintaining a healthy rose garden during the summer?"
    success_response = "For optimal summer care of your rose garden, start by watering deeply early in the morning to ensure the roots are well-hydrated without encouraging fungal growth. Apply a 2-3 inch layer of organic mulch around the base of the plants to conserve moisture and regulate soil temperature. Fertilize with a balanced rose fertilizer every 4–6 weeks to support healthy growth. Prune away any dead or diseased wood to promote good air circulation, and inspect regularly for pests such as aphids or spider mites, treating them promptly with an appropriate organic insecticidal soap. Finally, ensure that your roses receive at least 6 hours of direct sunlight daily for robust flowering."

    # Complex conversation example with tool calls
    complex_query = [
        {"role": "system", "content": "You are an expert in literature and can provide book recommendations."},
        {
            "createdAt": "2025-03-14T08:00:00Z",
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "I love historical fiction. Can you recommend a good book from that genre?",
                }
            ],
        },
    ]

    complex_response = [
        {
            "createdAt": "2025-03-14T08:00:05Z",
            "role": "assistant",
            "content": [{"type": "text", "text": "Let me fetch a recommendation for historical fiction."}],
        },
        {
            "createdAt": "2025-03-14T08:00:10Z",
            "role": "assistant",
            "content": [
                {
                    "type": "tool_call",
                    "tool_call_id": "tool_call_20250314_001",
                    "name": "get_book",
                    "arguments": {"genre": "historical fiction"},
                }
            ],
        },
        {
            "createdAt": "2025-03-14T08:00:15Z",
            "role": "tool",
            "tool_call_id": "tool_call_20250314_001",
            "content": [
                {
                    "type": "tool_result",
                    "tool_result": '{ "book": { "title": "The Pillars of the Earth", "author": "Ken Follett", "summary": "A captivating tale set in medieval England that weaves historical events with personal drama." } }',
                }
            ],
        },
        {
            "createdAt": "2025-03-14T08:00:20Z",
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": "Based on our records, I recommend 'The Pillars of the Earth' by Ken Follett. This novel is an excellent example of historical fiction with a rich narrative and well-developed characters. Would you like more details or another suggestion?",
                }
            ],
        },
    ]

    complex_tool_definitions = [
        {
            "name": "get_book",
            "description": "Retrieve a book recommendation for a specified genre.",
            "parameters": {
                "type": "object",
                "properties": {
                    "genre": {
                        "type": "string",
                        "description": "The genre for which a book recommendation is requested.",
                    }
                },
            },
        }
    ]

    print("Creating Eval Run with Inline Data")
    eval_run_object = client.evals.runs.create(
        eval_id=eval_object.id,
        name="inline_data_run",
        metadata={"team": "eval-exp", "scenario": "inline-data-v1"},
        data_source=CreateEvalJSONLRunDataSourceParam(
            type="jsonl",
            source=SourceFileContent(
                type="file_content",
                content=[
                    # Failure example - vague adherence
                    SourceFileContentContent(
                        item={"query": failure_query, "response": failure_response}
                    ),
                    # Success example - full adherence
                    SourceFileContentContent(
                        item={"query": success_query, "response": success_response}
                    ),
                    # Complex conversation example with tool calls
                    SourceFileContentContent(
                        item={
                            "query": complex_query,
                            "response": complex_response,
                            "tool_definitions": complex_tool_definitions,
                        }
                    ),
                ],
            ),
        ),
    )

    print("Eval Run created")
    pprint(eval_run_object)

    print("Get Eval Run by Id")
    eval_run_response = client.evals.runs.retrieve(run_id=eval_run_object.id, eval_id=eval_object.id)
    print("Eval Run Response:")
    pprint(eval_run_response)

    print("\n\n----Eval Run Output Items----\n\n")

    while True:
        run = client.evals.runs.retrieve(run_id=eval_run_response.id, eval_id=eval_object.id)
        if run.status in ("completed", "failed"):
            output_items = list(client.evals.runs.output_items.list(run_id=run.id, eval_id=eval_object.id))
            pprint(output_items)
            print(f"Eval Run Status: {run.status}")
            print(f"Eval Run Report URL: {run.report_url}")
            break
        time.sleep(5)
        print("Waiting for eval run to complete...")

Creating an OpenAI client from the AI Project client
Creating Evaluation
Evaluation created
Get Evaluation by Id
Eval Run Response:
EvalRetrieveResponse(id='eval_ea8d425455f5422091ecc98fbaf08a6e', created_at=1789483813, data_source_config=EvalCustomDataSourceConfig(schema_={'item': {'type': 'object', 'properties': {'query': {'anyOf': [{'type': 'string'}, {'type': 'array', 'items': {'type': 'object'}}]}, 'response': {'anyOf': [{'type': 'string'}, {'type': 'array', 'items': {'type': 'object'}}]}, 'tool_definitions': {'anyOf': [{'type': 'object'}, {'type': 'array', 'items': {'type': 'object'}}]}}, 'required': ['query', 'response']}, 'sample': {'type': 'object', 'properties': {'output_text': {'type': 'string'}}}}, type='custom', item_schema={}, include_sample_schema=True), metadata={}, name='Test Task Adherence Evaluator with inline data', object='eval', testing_criteria=[LabelModelGrader(input=None, labels=None, model=None, name='task_adherence', passing_labels=None, type='azure_ai_evalua